# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [51]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\n  OHLCV: {DATA_PATH_OHLCV}\n  Indices: {DATA_PATH_INDICES}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Paths Initialized.
  OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
  Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [52]:
# # === RELOADING FROM PARQUET ===
# features_df = pd.read_parquet("output/features_df.parquet")
# macro_df = pd.read_parquet("output/macro_df.parquet")
# df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
# df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
# df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

# df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
# df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
# df_fed = pd.read_parquet(
#     OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
# )

In [53]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features... takes about 6 minutes")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")

df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features... takes about 6 minutes
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1578, Days: 16188


In [54]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

## III. The Analysis Suite

In [55]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [56]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=10,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

DEBUG: 941 stocks passed filters on 2026-04-16


In [57]:
##################################
##################################
##################################

In [58]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 10))
[  4]   📂 tickers (len=10)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]   📈 initial_weights (shape=(10,))
[ 16]   📂 perf_metrics (len=24)
[ 17]     🔢 full_p_gain (float)
[ 18]     🔢 full_p_sharpe (float)
[ 19]     🔢 full_p_sharpe_atrp (float)
[ 20]     🔢 full_p_sharpe_trp (float)
[ 21]     🔢 lookback_p_gain (float)
[ 22]     🔢 lookback_p_sharpe (float)
[ 23]     🔢 lookback_p_sharpe_atrp (float)
[ 24]     🔢 lookback_p_sharpe_trp (float)
[ 25]     🔢 holding_p_gain (float)
[ 26]     🔢 holding_p_sharpe (float)
[ 27]     🔢 holding_p_sharpe_atrp (float)
[ 28]     🔢 holding_p_sharpe_tr

In [59]:
def list_all_paths(data):
    print(f"{'NAME':<25} | {'PATH'}")
    print("-" * 80)
    for item in data:
        # We use .get() to avoid errors if a specific dict is missing a key
        name = item.get("name", "N/A")
        path = item.get("path", "N/A")
        print(f"{name:<25} | {path}")


# Run the function
list_all_paths(result)

NAME                      | PATH
--------------------------------------------------------------------------------
audit_pack                | audit_pack
portfolio_series          | audit_pack -> portfolio_series
benchmark_series          | audit_pack -> benchmark_series
normalized_plot_data      | audit_pack -> normalized_plot_data
tickers                   | audit_pack -> tickers
index_0                   | audit_pack -> tickers -> index_0
index_1                   | audit_pack -> tickers -> index_1
index_2                   | audit_pack -> tickers -> index_2
index_3                   | audit_pack -> tickers -> index_3
index_4                   | audit_pack -> tickers -> index_4
index_5                   | audit_pack -> tickers -> index_5
index_6                   | audit_pack -> tickers -> index_6
index_7                   | audit_pack -> tickers -> index_7
index_8                   | audit_pack -> tickers -> index_8
index_9                   | audit_pack -> tickers -> index_9
initia

In [60]:
def get_value_by_key(search_term, data):
    """
    Search for a value by its name or its full path string.
    """
    for item in data:
        # Check if the search term matches the name OR the full path
        if item.get("name") == search_term or item.get("path") == search_term:
            return item.get("obj")

    return f"Error: Key '{search_term}' not found in the dataset."


# --- EXAMPLES OF USE ---

# 1. Accessing a simple metric (using name)
gain = get_value_by_key("full_p_gain", result)
print(f"Full Portfolio Gain: {gain}")

# 2. Accessing a specific dataframe (using path)
weights = get_value_by_key("audit_pack -> initial_weights", result)
print("\nInitial Weights Series:")
print(weights)

# 3. Accessing the results table
results_table = get_value_by_key("results_df", result)
print("\nTop Ranked Tickers:")
print(results_table)

Full Portfolio Gain: 0.5050820179921943

Initial Weights Series:
SOXL    0.1
CRDO    0.1
NBIS    0.1
IONQ    0.1
ALAB    0.1
BE      0.1
QBTS    0.1
CRWV    0.1
RVMD    0.1
RGTI    0.1
dtype: float64

Top Ranked Tickers:
        Rank  Strategy Value
Ticker                      
SOXL       1          0.5253
CRDO       2          0.5049
NBIS       3          0.4835
IONQ       4          0.4748
ALAB       5          0.4740
BE         6          0.4612
QBTS       7          0.4516
CRWV       8          0.4215
RVMD       9          0.4143
RGTI      10          0.3652


#### Independent Verification of create_walk_forward_analyzer Results

In [61]:
# Extracting the key dates
start_date = get_value_by_key("start_date", result)
decision_date = get_value_by_key("decision_date", result)
buy_date = get_value_by_key("buy_date", result)
holding_end_date = get_value_by_key("holding_end_date", result)

# Verification Print
print(f"Start Date:       {start_date}")
print(f"Decision Date:    {decision_date}")
print(f"Buy Date:         {buy_date}")
print(f"Holding End Date: {holding_end_date}")

Start Date:       2026-04-01 00:00:00
Decision Date:    2026-04-16 00:00:00
Buy Date:         2026-04-17 00:00:00
Holding End Date: 2026-04-24 00:00:00


In [62]:
target_tickers = get_value_by_key("tickers", result)
print(f"target_tickers: {target_tickers}")

target_tickers: ['SOXL', 'CRDO', 'NBIS', 'IONQ', 'ALAB', 'BE', 'QBTS', 'CRWV', 'RVMD', 'RGTI']


In [63]:
# --- Step 1: Extraction ---
# Extracting raw per-ticker components for manual weighting
raw_prices = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> prices", result
)
raw_atrp = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> atrp", result
)
raw_trp = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> trp", result
)

# Audit Pack expected values
metric_suffixes = ["gain", "sharpe", "sharpe_atrp", "sharpe_trp"]
period_prefixes = ["full", "lookback", "holding"]
audit_data = {
    f"{p}_p_{s}": get_value_by_key(f"{p}_p_{s}", result)
    for s in metric_suffixes
    for p in period_prefixes
}

# Metadata for display
metric_types = ["Log Gain", "Sharpe", "Sharpe (ATRP)", "Sharpe (TRP)"]
period_names = ["Full", "Lookback", "Holding"]
period_slices = {
    "Full": (start_date, holding_end_date),
    "Lookback": (start_date, decision_date),
    "Holding": (buy_date, holding_end_date),
}

portfolio_results = {}

# --- Step 2: Calculation Phase (Price-Drift Logic) ---
for name, (s_date, e_date) in period_slices.items():
    # A. Slice the wide-format data for this specific period
    p_slice = raw_prices.loc[s_date:e_date]
    a_slice = raw_atrp.loc[s_date:e_date]
    t_slice = raw_trp.loc[s_date:e_date]

    # B. Calculate Drifting Weights
    # Normalize prices so every ticker starts at 1.0 at the beginning of THIS slice
    norm_prices = p_slice / p_slice.iloc[0]
    # Weight_i = Normalized_Price_i / Sum(Normalized_Prices)
    weights = norm_prices.div(norm_prices.sum(axis=1), axis=0)

    # C. Calculate Portfolio Returns and Weighted Volatility
    # Equity curve is the simple average of normalized prices
    equity_curve = norm_prices.mean(axis=1)
    returns = equity_curve.pct_change().dropna()

    # Portfolio Volatility = Dot product of drifted weights and ticker volatility
    # We align the volatility indices with the returns index
    port_atrp = (weights * a_slice).sum(axis=1).loc[returns.index]
    port_trp = (weights * t_slice).sum(axis=1).loc[returns.index]

    # D. Final Metrics
    portfolio_results[f"{name} Log Gain"] = np.log(equity_curve.iloc[-1])
    portfolio_results[f"{name} Sharpe"] = (returns.mean() / returns.std()) * np.sqrt(
        252
    )
    portfolio_results[f"{name} Sharpe (ATRP)"] = returns.mean() / port_atrp.mean()
    portfolio_results[f"{name} Sharpe (TRP)"] = returns.mean() / port_trp.mean()

# --- Step 3: Display Phase ---
print(f"{'METRIC':<25} | {'CALCULATED':<12} | {'AUDIT PACK':<12} | {'MATCH?'}")
for m_type in metric_types:
    print("-" * 75)
    for p_name in period_names:
        key = f"{p_name} {m_type}"
        # Map display name to audit key
        sfx = m_type.lower().replace(" ", "_").replace("(", "").replace(")", "")
        audit_key = f"{p_name.lower()}_p_{'gain' if sfx == 'log_gain' else sfx}"

        calc_val, audit_val = portfolio_results[key], audit_data[audit_key]
        is_match = np.isclose(calc_val, audit_val, rtol=1e-05)
        print(
            f"{key:<25} | {calc_val:<12.6f} | {audit_val:<12.6f} | {'✅ YES' if is_match else '❌ NO'}"
        )

# --- Step 4: Assertions ---
for key, val in portfolio_results.items():
    p, m = key.split(" ", 1)
    sfx = m.lower().replace(" ", "_").replace("(", "").replace(")", "")
    assert np.isclose(
        val,
        audit_data[f"{p.lower()}_p_{'gain' if sfx == 'log_gain' else sfx}"],
        rtol=1e-05,
    )

print(
    "-" * 75
    + "\nVerification complete. Price-drift weights corrected the Holding period."
)

METRIC                    | CALCULATED   | AUDIT PACK   | MATCH?
---------------------------------------------------------------------------
Full Log Gain             | 0.505082     | 0.505082     | ✅ YES
Lookback Log Gain         | 0.458616     | 0.458616     | ✅ YES
Holding Log Gain          | 0.030589     | 0.030589     | ✅ YES
---------------------------------------------------------------------------
Full Sharpe               | 14.007071    | 14.007071    | ✅ YES
Lookback Sharpe           | 19.603917    | 19.603917    | ✅ YES
Holding Sharpe            | 5.402948     | 5.402948     | ✅ YES
---------------------------------------------------------------------------
Full Sharpe (ATRP)        | 0.468076     | 0.468076     | ✅ YES
Lookback Sharpe (ATRP)    | 0.648441     | 0.648441     | ✅ YES
Holding Sharpe (ATRP)     | 0.097947     | 0.097947     | ✅ YES
---------------------------------------------------------------------------
Full Sharpe (TRP)         | 0.400320     | 0.400320    

In [64]:
##################################
##################################
##################################

In [65]:
# 1. Setup Parameters
# NVDA had a dividend payment date on 2026-04-01
verification_ticker = "NVDA"
# Changed to the last date shown in your features_df.info()
target_date = pd.Timestamp("2026-04-24")
lookback_days = 120

# 2. Extract Data Safely
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")

# --- DATE VALIDATION ---
# If our chosen target_date is not in the index, grab the most recent available one
if target_date not in ticker_prices.index:
    new_target = ticker_prices.index[ticker_prices.index <= target_date][-1]
    print(
        f"Adjusting target date from {target_date.date()} to {new_target.date()} (Last available trading day)"
    )
    target_date = new_target

# Slice the previous 100+ days for rolling calculations
start_date_slice = target_date - pd.Timedelta(days=lookback_days)
raw_slice = ticker_prices.loc[:target_date].tail(90).copy()  # Get enough history
feat_row = ticker_features.loc[target_date]

# --- 3. Manual Calculations (The "Easy" Group) ---

# A. Ret_1d: (Current / Previous) - 1
manual_ret_1d = raw_slice["Adj Close"].pct_change().iloc[-1]

# B. Mom_21: (Current / Price 21-trading-bars ago) - 1
manual_mom_21 = raw_slice["Adj Close"].pct_change(21).iloc[-1]

# C. DD_21: (Current / Max in last 21 trading bars) - 1
rolling_max_21 = raw_slice["Adj Close"].rolling(window=21).max()
manual_dd_21 = (raw_slice["Adj Close"] / rolling_max_21 - 1).iloc[-1]

# D. Consistency: % of last 5 trading days where return was positive
daily_rets = raw_slice["Adj Close"].pct_change()
manual_consist = (daily_rets > 0).rolling(window=5).mean().iloc[-1]

# E. TRP (True Range Percent): Raw daily volatility
prev_close = raw_slice["Adj Close"].shift(1)
tr_val = pd.concat(
    [
        (raw_slice["Adj High"] - raw_slice["Adj Low"]),
        (raw_slice["Adj High"] - prev_close).abs(),
        (raw_slice["Adj Low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)
manual_trp = (tr_val / raw_slice["Adj Close"]).iloc[-1]

# 4. Comparison Table
easy_verification = pd.DataFrame(
    {
        "Feature": ["Ret_1d", "Mom_21", "DD_21", "Consistency", "TRP"],
        "Manual_Value": [
            manual_ret_1d,
            manual_mom_21,
            manual_dd_21,
            manual_consist,
            manual_trp,
        ],
        "Feature_DF_Value": [
            feat_row["Ret_1d"],
            feat_row["Mom_21"],
            feat_row["DD_21"],
            feat_row["Consistency"],
            feat_row["TRP"],
        ],
    }
)

easy_verification["Abs_Error"] = (
    easy_verification["Manual_Value"] - easy_verification["Feature_DF_Value"]
).abs()

print(f"\nVerification for {verification_ticker} on {target_date.date()}")
print(
    easy_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:.6f}".format,
            "Feature_DF_Value": "{:.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if easy_verification["Abs_Error"].max() < 1e-10:
    print("\n✅ Easy Group Verified.")
else:
    print("\n❌ Discrepancy detected.")


Verification for NVDA on 2026-04-24
    Feature Manual_Value Feature_DF_Value Abs_Error
     Ret_1d     0.043228         0.043228  0.00e+00
     Mom_21     0.165603         0.165603  0.00e+00
      DD_21     0.000000         0.000000  0.00e+00
Consistency     0.600000         0.600000  0.00e+00
        TRP     0.054305         0.054305  0.00e+00

✅ Easy Group Verified.


In [66]:
# 1. Setup Data for NVDA (Using the full history for smoothing warm-up)
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

# For smoothing, we need a long history to match Wilder's logic exactly
adj_close = ticker_prices["Adj Close"]
adj_high = ticker_prices["Adj High"]
adj_low = ticker_prices["Adj Low"]
rsi_period = GLOBAL_SETTINGS["rsi_period"]  # usually 14
atr_period = GLOBAL_SETTINGS["atr_period"]  # usually 14

# --- 2. Manual Calculations (Group 2) ---

# A. RSI (Wilder's Smoothing)
delta = adj_close.diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

# Wilder's smoothing is an EWM with alpha = 1/period
avg_gain = gain.ewm(alpha=1 / rsi_period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1 / rsi_period, adjust=False).mean()

rs = avg_gain / avg_loss
manual_rsi = (100 - (100 / (1 + rs))).loc[target_date]

# B. ATRP (Average True Range Percent)
# First get True Range (TR)
prev_close = adj_close.shift(1)
tr = pd.concat(
    [(adj_high - adj_low), (adj_high - prev_close).abs(), (adj_low - prev_close).abs()],
    axis=1,
).max(axis=1)

# ATR is Wilder's smoothing of TR
manual_atr = tr.ewm(alpha=1 / atr_period, adjust=False).mean()
manual_atrp = (manual_atr / adj_close).loc[target_date]

# C. AutoCorr_15 (1-day lag correlation)
rets = adj_close.pct_change()
# Calculate rolling correlation of returns vs lagged returns
manual_autocorr = rets.rolling(window=15).corr(rets.shift(1)).loc[target_date]

# --- 3. Comparison Table ---
group2_verification = pd.DataFrame(
    {
        "Feature": ["RSI", "ATRP", "AutoCorr_15"],
        "Manual_Value": [manual_rsi, manual_atrp, manual_autocorr],
        "Feature_DF_Value": [
            feat_row["RSI"],
            feat_row["ATRP"],
            feat_row["AutoCorr_15"],
        ],
    }
)

group2_verification["Abs_Error"] = (
    group2_verification["Manual_Value"] - group2_verification["Feature_DF_Value"]
).abs()

print(f"Verification for {verification_ticker} on {target_date.date()}")
print(
    group2_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:.6f}".format,
            "Feature_DF_Value": "{:.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if group2_verification["Abs_Error"].max() < 1e-8:
    print("\n✅ Smoothed Group Verified.")
else:
    print(
        "\n❌ Discrepancy detected. This often happens if the 'QuantUtils' uses a different smoothing initialization than 'adjust=False'."
    )

Verification for NVDA on 2026-04-24
    Feature Manual_Value Feature_DF_Value Abs_Error
        RSI    71.498227        71.498227  0.00e+00
       ATRP     0.026167         0.026167  0.00e+00
AutoCorr_15    -0.371361        -0.371361  0.00e+00

✅ Smoothed Group Verified.


In [67]:
# 1. Setup Parameters from GLOBAL_SETTINGS
# We use the 252-day window as confirmed by our previous diagnostic
verification_ticker = "NVDA"
target_date = pd.Timestamp("2026-04-24")
q_window = GLOBAL_SETTINGS["quality_window"]  # 252 days
q_min = GLOBAL_SETTINGS["quality_min_periods"]  # 126 days

# 2. Extract Slices
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

# --- 3. MANUAL CALCULATIONS ---

# A. Data Health Metrics (Staleness & Frozen Volume)
# Logic: Identifies low-quality data or "zombie" tickers
is_stale = (
    (ticker_prices["Volume"] == 0)
    | (ticker_prices["Adj High"] == ticker_prices["Adj Low"])
).astype(int)
manual_stale_pct = (
    is_stale.rolling(window=q_window, min_periods=q_min).mean().loc[target_date]
)

has_same_vol = (ticker_prices["Volume"].diff() == 0).astype(int)
manual_same_vol = (
    has_same_vol.rolling(window=q_window, min_periods=q_min).sum().loc[target_date]
)

# B. Liquidity Metric (RollMedDollarVol)
# Logic: Measures typical daily dollar turnover over 1 year (252 days).
# Using Median instead of Mean protects the filter from being skewed by single-day volume spikes.
dollar_vol = ticker_prices["Adj Close"] * ticker_prices["Volume"]
manual_med_dvol = (
    dollar_vol.rolling(window=q_window, min_periods=q_min).median().loc[target_date]
)

# C. Trend Acceleration (Convexity)
# Logic: Calculated as the 2-day change in the 5-day price slope.
# A positive value indicates the trend is accelerating upward (Parabolic).
# A negative value indicates the trend is decelerating (Exhaustion).
s_idx = ticker_features.index.get_loc(target_date)
slope_t = ticker_features["Slope_P_5"].iloc[s_idx]
slope_t_minus_2 = ticker_features["Slope_P_5"].iloc[s_idx - 2]
manual_convexity = slope_t - slope_t_minus_2


# --- 4. VERIFICATION OUTPUT ---

final_verification = pd.DataFrame(
    {
        "Feature": [
            "RollingStalePct",
            "RollingSameVolCount",
            "RollMedDollarVol",
            "Convexity",
        ],
        "Manual_Value": [
            manual_stale_pct,
            manual_same_vol,
            manual_med_dvol,
            manual_convexity,
        ],
        "Feature_DF_Value": [
            feat_row["RollingStalePct"],
            feat_row["RollingSameVolCount"],
            feat_row["RollMedDollarVol"],
            feat_row["Convexity"],
        ],
    }
)

# Calculate error and formatting
final_verification["Abs_Error"] = (
    final_verification["Manual_Value"] - final_verification["Feature_DF_Value"]
).abs()

print(f"--- Final Group Verification: {verification_ticker} ({target_date.date()}) ---")
print(
    final_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:,.4f}".format,
            "Feature_DF_Value": "{:,.4f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

# Assertion Check
if final_verification["Abs_Error"].max() < 1e-7:
    print("\n✅ Final Group Verified: All quality and acceleration metrics match.")
else:
    print("\n❌ Discrepancy detected in final group.")

--- Final Group Verification: NVDA (2026-04-24) ---
            Feature        Manual_Value    Feature_DF_Value Abs_Error
    RollingStalePct              0.0000              0.0000  0.00e+00
RollingSameVolCount              0.0000              0.0000  0.00e+00
   RollMedDollarVol 29,915,329,730.9855 29,915,329,730.9855  0.00e+00
          Convexity              0.0027              0.0027  0.00e+00

✅ Final Group Verified: All quality and acceleration metrics match.


In [68]:
# 1. Setup Data for Audit
verification_ticker = "NVDA"
target_date = pd.Timestamp("2026-04-24")
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

# Window constants from GLOBAL_SETTINGS
atr_period = GLOBAL_SETTINGS["atr_period"]  # 14
win_63 = GLOBAL_SETTINGS["63d_window"]  # 63
win_20 = 20  # Hardcoded in Range_Pos logic

# Slicing context
s_idx = ticker_prices.index.get_loc(target_date)
adj_close = ticker_prices["Adj Close"]
adj_high = ticker_prices["Adj High"]
adj_low = ticker_prices["Adj Low"]
volume = ticker_prices["Volume"]


# Helper: Weighted Slope Kernel confirmed in previous steps
def apply_5d_slope_kernel(series, idx):
    y = series.iloc[idx - 4 : idx + 1].values
    weights = np.array([-2, -1, 0, 1, 2])
    return np.sum(y * weights) / 10


# --- 2. MANUAL CALCULATIONS ---

# A. ATR (Average True Range)
# Logic: Wilder's smoothing of the True Range.
# Foundational for risk-adjusted position sizing.
prev_close = adj_close.shift(1)
tr = pd.concat(
    [(adj_high - adj_low), (adj_high - prev_close).abs(), (adj_low - prev_close).abs()],
    axis=1,
).max(axis=1)
manual_atr = tr.ewm(alpha=1 / atr_period, adjust=False).mean().loc[target_date]

# B. Range_Pos_20 (Price Channel Position)
# Logic: Where the current price sits relative to the 20-day High/Low.
# 1.0 = at 20-day high, 0.0 = at 20-day low.
h_20 = adj_high.rolling(window=win_20).max().loc[target_date]
l_20 = adj_low.rolling(window=win_20).min().loc[target_date]
manual_range_pos = (adj_close.loc[target_date] - l_20) / (h_20 - l_20)

# C. Beta_63 & IR_63 (Relative Market Performance)
stock_rets = adj_close.pct_change()

# Align indices by intersection to prevent KeyError
common_idx = stock_rets.index.intersection(macro_df.index)
mkt_rets = macro_df.loc[common_idx, "Mkt_Ret"]
stock_rets_aligned = stock_rets.loc[common_idx]

# Recalculate s_idx based on the aligned series
s_idx_aligned = stock_rets_aligned.index.get_loc(target_date)

y = stock_rets_aligned.iloc[s_idx_aligned - 62 : s_idx_aligned + 1]
x_mkt = mkt_rets.iloc[s_idx_aligned - 62 : s_idx_aligned + 1]

# Beta calculation
manual_beta_63 = np.cov(y, x_mkt)[0, 1] / np.var(x_mkt, ddof=1)

# IR calculation (Ensure 'excess_rets' is defined or use local 'excess' variable)
excess = y - x_mkt
manual_ir_63 = excess.mean() / excess.std()


# D. Slope_P_5 (Price Trend Velocity)
# Logic: 5-day OLS slope of Log Prices (continuous growth rate).
manual_slope_p = apply_5d_slope_kernel(np.log(adj_close), s_idx)

# Updated Manual Calculation for E (Slope_V_5)
v_baseline = (
    volume.rolling(window=win_63, min_periods=1).mean().replace(0, 1e-8)
)  # MATCH PRODUCTION
v_rel = volume / v_baseline
# Ensure we handle NaNs in cumsum exactly like production
obv_rel = (np.sign(adj_close.diff()).fillna(0) * v_rel).cumsum()
manual_slope_v = apply_5d_slope_kernel(obv_rel, s_idx)


# --- 3. AUDIT SUMMARY TABLE ---

audit_results = pd.DataFrame(
    {
        "Feature": [
            "ATR",
            "IR_63",
            "Beta_63",
            "Range_Pos_20",
            "Slope_P_5",
            "Slope_V_5",
        ],
        "Manual_Value": [
            manual_atr,
            manual_ir_63,
            manual_beta_63,
            manual_range_pos,
            manual_slope_p,
            manual_slope_v,
        ],
        "Feature_DF_Value": [
            feat_row["ATR"],
            feat_row["IR_63"],
            feat_row["Beta_63"],
            feat_row["Range_Pos_20"],
            feat_row["Slope_P_5"],
            feat_row["Slope_V_5"],
        ],
    }
)

audit_results["Abs_Error"] = (
    audit_results["Manual_Value"] - audit_results["Feature_DF_Value"]
).abs()

print(f"--- Completion Audit: {verification_ticker} ({target_date.date()}) ---")
print(
    audit_results.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:,.6f}".format,
            "Feature_DF_Value": "{:,.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if audit_results["Abs_Error"].max() < 1e-7:
    print("\n✅ All remaining columns verified. Audit Complete.")
else:
    print("\n❌ Audit mismatch detected in remaining columns.")

--- Completion Audit: NVDA (2026-04-24) ---
     Feature Manual_Value Feature_DF_Value Abs_Error
         ATR     5.449874         5.449874  0.00e+00
       IR_63     0.072796         0.072796  1.18e-15
     Beta_63     1.813462         1.813462  6.64e-14
Range_Pos_20     0.942588         0.942588  0.00e+00
   Slope_P_5     0.005934         0.005934  0.00e+00
   Slope_V_5     0.111707         0.111707  0.00e+00

✅ All remaining columns verified. Audit Complete.


In [69]:
#########################
#########################
#########################

In [70]:
# 1. Setup Audit for the whole Universe on one specific day
target_date = pd.Timestamp("2026-04-24")
verification_ticker = "NVDA"

# 2. Get the "Cross-Section" (All tickers for this date)
# We pull the raw convexity values for every ticker from features_df
universe_convexity = features_df.xs(target_date, level="Date")["Convexity"]

# 3. Manual "Engine" Logic (Replicating get_agent_view)
# Step A: Clean (Remove Inf/NaN)
clean_universe = universe_convexity.replace([np.inf, -np.inf], np.nan).dropna()

# Step B: Manual Z-Score
u_mean = clean_universe.mean()
u_std = clean_universe.std()
manual_z_scores = (clean_universe - u_mean) / u_std

# Step C: Neutralize & Clip (Filling NaNs with 0 as per .fillna(0).clip())
clip_val = GLOBAL_SETTINGS.get("feature_zscore_clip", 4.0)
final_agent_values = manual_z_scores.fillna(0).clip(-clip_val, clip_val)

# 4. Compare for NVDA
raw_val = clean_universe.loc[verification_ticker]
agent_val = final_agent_values.loc[verification_ticker]

print(f"--- Pillar 6 (Convexity) Cross-Sectional Audit ---")
print(f"Date: {target_date.date()} | Universe Size: {len(clean_universe)}")
print(f"Universe Mean Convexity: {u_mean:.6f}")
print(f"Universe Std Convexity : {u_std:.6f}")
print("-" * 50)
print(f"NVDA Raw Convexity    : {raw_val:.6f}")
print(f"NVDA Agent View (Z)   : {agent_val:.6f}")

# verification against the registry output (if you have the registry object 'pillar_6')
# pillar_6_view = pillar_6.get_agent_view(obs_for_that_day)

--- Pillar 6 (Convexity) Cross-Sectional Audit ---
Date: 2026-04-24 | Universe Size: 1578
Universe Mean Convexity: -0.003388
Universe Std Convexity : 0.011957
--------------------------------------------------
NVDA Raw Convexity    : 0.002689
NVDA Agent View (Z)   : 0.508300


In [74]:
from core.contracts import MetricBlueprint
from types import SimpleNamespace

# 1. Prepare the "obs" object that the Blueprint expects
# We take the features for ALL tickers on our target date
daily_snapshot = features_df.xs(target_date, level="Date")

# The Blueprint formula expects attributes like 'obs.convexity'
# We create a SimpleNamespace where columns are attributes
obs = SimpleNamespace(
    convexity=daily_snapshot["Convexity"],
    slope_p_5=daily_snapshot["Slope_P_5"],
    slope_v_5=daily_snapshot["Slope_V_5"],
)

# 2. Re-create the Blueprint exactly as it is in your registry.py
convexity_blueprint = MetricBlueprint(
    name="Convexity",
    category="Physics",
    regime="Acceleration",
    description="Second derivative of price.",
    agent_hint="The 'Golden Exit'.",
    intervention_trigger="EXIT LONG if Value < -0.7",
    scaling_type="Z-Score",
    formula=lambda obs: obs.convexity,
)

# 3. GET THE SYSTEM OUTPUT
# This runs the actual Engine logic: Clean -> Scale -> Clip
system_output_series = convexity_blueprint.get_agent_view(obs)
system_val = system_output_series.loc[verification_ticker]

# 4. FINAL VERIFICATION COMPARISON
print(f"--- Final System Verification: Pillar 6 ---")
print(f"Manual Audit Value (Z):  {agent_val:.5f}")
print(f"System Output Value (Z): {system_val:.5f}")

error = abs(agent_val - system_val)
if error < 1e-5:
    print(
        f"\n✅ VERIFIED: The System Logic matches the Manual Audit (Error: {error:.2e})"
    )
else:
    print(f"\n❌ DISCREPANCY: System output differs from manual calculation!")

--- Final System Verification: Pillar 6 ---
Manual Audit Value (Z):  0.50830
System Output Value (Z): 0.50830

✅ VERIFIED: The System Logic matches the Manual Audit (Error: 0.00e+00)


In [76]:
import pandas as pd
import numpy as np
from types import SimpleNamespace


# --- 1. Mock QuantUtils to match the Registry's expected environment ---
class QuantUtils:
    @staticmethod
    def zscore(series: pd.Series) -> pd.Series:
        """Robust Z-score handling NaNs/Infs for cross-sectional data."""
        clean = series.replace([np.inf, -np.inf], np.nan)
        return (clean - clean.mean()) / clean.std()


# --- 2. Prepare the Observation (obs) for 2026-04-24 ---
target_date = pd.Timestamp("2026-04-24")
verification_ticker = "NVDA"
daily_snapshot = features_df.xs(target_date, level="Date")

obs = SimpleNamespace(
    slope_v_5=daily_snapshot["Slope_V_5"], slope_p_5=daily_snapshot["Slope_P_5"]
)

# --- 3. RE-CREATE BLUEPRINT (Exactly from registry.py) ---
divergence_blueprint = MetricBlueprint(
    name="OBV Divergence (5d)",
    category="Volume/Fuel",
    regime="Confirmation",
    description="Z-scored gap between relative volume flow and price trend.",
    agent_hint="Detects smart money accumulation/distribution.",
    intervention_trigger="INVALIDATE Longs if Divergence < -1.0std.",
    scaling_type="Z-Score",  # Triggers SECOND Z-score in get_agent_view
    formula=lambda obs: (
        QuantUtils.zscore(obs.slope_v_5) - QuantUtils.zscore(obs.slope_p_5)
    ),
)

# --- 4. MANUAL AUDIT (Step-by-Step Logic) ---

# Step A: Internal Formula Math (First Z-score)
z_vol = QuantUtils.zscore(obs.slope_v_5)
z_price = QuantUtils.zscore(obs.slope_p_5)
raw_divergence = z_vol - z_price

# Step B: Engine Scaling Math (Second Z-score)
# get_agent_view() cleans the raw output and Z-scores it AGAIN
final_manual_z = QuantUtils.zscore(raw_divergence)

# Step C: Neutralize & Clip
clip_val = GLOBAL_SETTINGS.get("feature_zscore_clip", 4.0)
manual_final_val = (
    final_manual_z.fillna(0).clip(-clip_val, clip_val).loc[verification_ticker]
)

# --- 5. SYSTEM COMPARISON ---
system_output_series = divergence_blueprint.get_agent_view(obs)
system_final_val = system_output_series.loc[verification_ticker]

# --- 6. RESULTS ---
print(f"--- Pillar 5 (OBV Divergence) Completion Audit ---")
print(f"NVDA Z-Volume (Internal): {z_vol.loc[verification_ticker]:10.5f}")
print(f"NVDA Z-Price  (Internal): {z_price.loc[verification_ticker]:10.5f}")
print(f"NVDA Raw Div (V_z - P_z): {raw_divergence.loc[verification_ticker]:10.5f}")
print("-" * 50)
print(f"Manual Final Z-Score    : {manual_final_val:10.5f}")
print(f"System Final Z-Score    : {system_final_val:10.5f}")

error = abs(manual_final_val - system_final_val)
if error < 1e-5:
    print(
        f"\n✅ VERIFIED: Pillar 5 logic is mathematically consistent (Error: {error:.2e})"
    )
    print("The 'Double Z-Score' architecture is working as designed.")
else:
    print(f"\n❌ DISCREPANCY: Check QuantUtils.zscore implementation details.")

--- Pillar 5 (OBV Divergence) Completion Audit ---
NVDA Z-Volume (Internal):    0.48328
NVDA Z-Price  (Internal):    0.71017
NVDA Raw Div (V_z - P_z):   -0.22690
--------------------------------------------------
Manual Final Z-Score    :   -0.31292
System Final Z-Score    :   -0.31292

✅ VERIFIED: Pillar 5 logic is mathematically consistent (Error: 0.00e+00)
The 'Double Z-Score' architecture is working as designed.
